# H2 Feature Engineering: Preseason Rating and Tournament Success

H2 tests whether adding preseason rating to regular-season win rate improves predictions of tournament success.

| Data source | Use |
|---|---|
| `regular_season_only.csv.gz` | Calculate each team’s regular-season wins, games played, and win rate. |
| `team_strength_preseason.csv` | Add each team’s preseason rating. |
| `team_conferences.csv` | Identify the teams active in each season, including teams that did not enter the tournament. |
| `tournament_results.csv` | Determine whether each team won at least one main-bracket game. |

We use seasons 2015–2019 and 2021–2025 for labeled data. We omit 2020 because there was no tournament, and prepare 2026 separately for forecasts because its tournament outcome is not available.

Teams that did not enter the tournament, and teams that entered but won no main-bracket games, receive an outcome of 0.

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = (
    Path.home()
    / "Documents"
    / "GitHub"
    / "CSC-4913-Project-4"
    / "cleaned_data"
)

PROJECT_DIR = DATA_DIR.parent

print("Data folder:", DATA_DIR)
print("Folder found:", DATA_DIR.exists())


def read_cleaned_csv(filename):
    path = DATA_DIR / filename

    # File Explorer may hide the final .gz extension.
    if not path.exists() and filename.endswith(".csv.gz"):
        path = DATA_DIR / filename[:-3]

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {filename} in {DATA_DIR}. "
            f"Files found: {[p.name for p in DATA_DIR.iterdir()]}"
        )

    return pd.read_csv(path)


regular_games = read_cleaned_csv("regular_season_only.csv.gz")
ratings = read_cleaned_csv("team_strength_preseason.csv")
team_conferences = read_cleaned_csv("team_conferences.csv")
tournament_results = read_cleaned_csv("tournament_results.csv")

labeled_seasons = list(range(2015, 2020)) + list(range(2021, 2026))
feature_seasons = labeled_seasons + [2026]

regular_games = regular_games[
    regular_games["season"].isin(feature_seasons)
].copy()

ratings = ratings[
    ratings["season"].isin(feature_seasons)
].copy()

team_conferences = team_conferences[
    team_conferences["season"].isin(feature_seasons)
].copy()

tournament_results = tournament_results[
    tournament_results["season"].isin(labeled_seasons)
].copy()

print("Regular-season games by season:")
display(
    regular_games.groupby("season")
    .size()
    .rename("game_rows")
    .reset_index()
)

print("Tournament result seasons:", sorted(tournament_results["season"].unique()))

Data folder: C:\Users\lsadi\Documents\GitHub\CSC-4913-Project-4\cleaned_data
Folder found: True
Regular-season games by season:


,season,game_rows
0,2015,5202
1,2016,5223
2,2017,5218
3,2018,5247
4,2019,5260
5,2021,3767
6,2022,5276
7,2023,5274
8,2024,5277
9,2025,5288


Tournament result seasons: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


### Observation unit and predictors

Each row represents one active team in one season. Regular-season win rate is calculated as wins divided by games played.

| Column | Meaning | Model role |
|---|---|---|
| `season` | Season | Determines the training, validation, or test split |
| `team_id` | Team represented by the row | Identifies the team |
| `regular_season_wins` | Regular-season wins | Used to calculate win rate |
| `regular_season_games` | Regular-season games played | Used to calculate win rate |
| `regular_season_win_rate` | Wins divided by games played | Predictor in both models |
| `preseason_rating` | Team’s preseason rating | Additional predictor in the second model |
| `won_at_least_one_main_bracket_game` | Whether the team won at least one main-bracket game | Outcome |

The baseline model uses regular-season win rate. The second model uses regular-season win rate and preseason rating. The outcome is not included among the predictors.

In [2]:
# Make one win/loss record for each team in each regular-season game.
wins = (
    regular_games[["season", "w_team_id"]]
    .rename(columns={"w_team_id": "team_id"})
    .assign(win=1)
)

losses = (
    regular_games[["season", "l_team_id"]]
    .rename(columns={"l_team_id": "team_id"})
    .assign(win=0)
)

team_game_results = pd.concat([wins, losses], ignore_index=True)

team_season_stats = (
    team_game_results
    .groupby(["season", "team_id"], as_index=False)
    .agg(
        regular_season_wins=("win", "sum"),
        regular_season_games=("win", "size"),
    )
)

team_season_stats["regular_season_win_rate"] = (
    team_season_stats["regular_season_wins"]
    / team_season_stats["regular_season_games"]
)

# Use active team-season pairs so the table includes teams that did not
# enter the tournament, but not teams from seasons when they were inactive.
active_team_seasons = (
    team_conferences[["season", "team_id"]]
    .drop_duplicates(["season", "team_id"])
)

rating_features = ratings[
    ["season", "team_id", "preseason_rating"]
].drop_duplicates(["season", "team_id"])

h2_features = (
    active_team_seasons
    .merge(
        team_season_stats,
        on=["season", "team_id"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        rating_features,
        on=["season", "team_id"],
        how="left",
        validate="one_to_one",
    )
)

missing_features = h2_features[
    [
        "regular_season_wins",
        "regular_season_games",
        "regular_season_win_rate",
        "preseason_rating",
    ]
].isna().sum()

if missing_features.any():
    raise ValueError(f"Some team-season features are missing:\n{missing_features}")

print("Sample team-season feature rows:")
display(h2_features.head(8))

Sample team-season feature rows:


,season,team_id,regular_season_wins,regular_season_games,regular_season_win_rate,preseason_rating
0,2015,1101,14,30,0.466667,5.8
1,2015,1102,13,30,0.433333,-4.0
2,2015,1103,17,30,0.566667,4.1
3,2015,1104,23,31,0.741935,6.5
4,2015,1105,27,30,0.900000,15.2
5,2015,1106,13,27,0.481481,-0.1
6,2015,1107,14,31,0.451613,-6.3
7,2015,1108,17,27,0.629630,-9.1


### Define tournament success

The outcome is 1 if a team won at least one main-bracket game and 0 otherwise. A team receives 0 if it did not enter the tournament or entered but did not win a game.

The 2026 feature rows are kept separate because no 2026 tournament outcome is available. They will be used for forecasts, not scored predictions.

In [3]:
target_column = "won_at_least_one_main_bracket_game"

main_bracket_results = tournament_results[
    tournament_results["day_num"] >= 136
]

tournament_winners = set(
    zip(main_bracket_results["season"], main_bracket_results["w_team_id"])
)

h2_labeled = h2_features[
    h2_features["season"].isin(labeled_seasons)
].copy()

h2_labeled[target_column] = [
    int((season, team_id) in tournament_winners)
    for season, team_id in zip(
        h2_labeled["season"],
        h2_labeled["team_id"],
    )
]

# Keep 2026 predictors separate; do not create a tournament label for 2026.
h2_2026_forecast_features = h2_features[
    h2_features["season"].eq(2026)
].copy()

if target_column in h2_2026_forecast_features.columns:
    raise ValueError("2026 forecast rows should not contain a tournament outcome.")

print("Labeled team-season rows and positive outcomes by season:")
display(
    h2_labeled.groupby("season")
    .agg(
        team_seasons=("team_id", "size"),
        teams_with_at_least_one_win=(target_column, "sum"),
    )
    .reset_index()
)

print("2026 forecast rows for Team A and Team C:")
display(
    h2_2026_forecast_features[
        h2_2026_forecast_features["team_id"].isin([1105, 1120])
    ]
)

Labeled team-season rows and positive outcomes by season:


,season,team_seasons,teams_with_at_least_one_win
0,2015,355,32
1,2016,355,32
2,2017,356,32
3,2018,357,32
4,2019,358,32
5,2021,359,32
6,2022,360,32
7,2023,360,32
8,2024,360,32
9,2025,360,32


2026 forecast rows for Team A and Team C:


,season,team_id,regular_season_wins,regular_season_games,regular_season_win_rate,preseason_rating
3584,2026,1105,21,30,0.700000,17.2
3599,2026,1120,24,28,0.857143,3.6


### Walk-forward validation and final test

The initial training seasons are 2015–2019. We validate on 2021, then add each completed validation season to the training data before validating on the next season through 2024. The final test is 2025.

After evaluating the selected model on 2025, the model can be refit on labeled seasons through 2025 to produce unscored 2026 forecasts.

Both logistic-regression versions use the same folds. Any scaling for logistic regression should be fit on the training portion of each fold only.

In [4]:
fold_specs = []
training_seasons = list(range(2015, 2020))

# Walk-forward validation folds: 2021–2024.
for validation_season in range(2021, 2025):
    fold_specs.append({
        "fold": f"validation_{validation_season}",
        "training_seasons": training_seasons.copy(),
        "heldout_season": validation_season,
        "heldout_role": "validation",
    })
    training_seasons.append(validation_season)

# Final scored test: train through 2024, hold out 2025.
final_test_training_seasons = (
    list(range(2015, 2020)) + list(range(2021, 2025))
)

fold_specs.append({
    "fold": "final_test_2025",
    "training_seasons": final_test_training_seasons,
    "heldout_season": 2025,
    "heldout_role": "test",
})

fold_tables = []
manifest_rows = []

for spec in fold_specs:
    included_seasons = spec["training_seasons"] + [spec["heldout_season"]]

    fold_data = h2_labeled[
        h2_labeled["season"].isin(included_seasons)
    ].copy()

    fold_data["fold"] = spec["fold"]
    fold_data["data_role"] = "train"
    fold_data.loc[
        fold_data["season"].eq(spec["heldout_season"]),
        "data_role",
    ] = spec["heldout_role"]

    fold_tables.append(fold_data)

    manifest_rows.append({
        "fold": spec["fold"],
        "training_seasons": ",".join(map(str, spec["training_seasons"])),
        "heldout_season": spec["heldout_season"],
        "heldout_role": spec["heldout_role"],
    })

h2_fold_data = pd.concat(fold_tables, ignore_index=True)
h2_fold_manifest = pd.DataFrame(manifest_rows)

# These are the two feature sets for the logistic-regression comparison.
baseline_features = ["regular_season_win_rate"]
rating_added_features = ["regular_season_win_rate", "preseason_rating"]

# For the unscored 2026 forecast, refit using labeled seasons through 2025.
forecast_training_seasons = labeled_seasons.copy()

output_dir = PROJECT_DIR / "prepared_data" / "H2"
output_dir.mkdir(parents=True, exist_ok=True)

h2_labeled.to_csv(output_dir / "h2_team_season_labeled.csv", index=False)
h2_fold_data.to_csv(output_dir / "h2_fold_data.csv", index=False)
h2_fold_manifest.to_csv(output_dir / "h2_fold_manifest.csv", index=False)
h2_2026_forecast_features.to_csv(
    output_dir / "h2_2026_forecast_features.csv",
    index=False,
)

print("Fold manifest:")
display(h2_fold_manifest)

print("Final test sample (2025):")
display(
    h2_fold_data[
        h2_fold_data["fold"].eq("final_test_2025")
        & h2_fold_data["data_role"].eq("test")
    ][[
        "season",
        "team_id",
        "regular_season_win_rate",
        "preseason_rating",
        target_column,
        "data_role",
    ]].head(8)
)

print("Baseline predictors:", baseline_features)
print("Rating-added predictors:", rating_added_features)
print("2026 forecast training seasons:", forecast_training_seasons)
print("Saved H2 tables to:", output_dir)

Fold manifest:


,fold,training_seasons,heldout_season,heldout_role
0,validation_2021,"2015,2016,2017,2018,2019",2021,validation
1,validation_2022,"2015,2016,2017,2018,2019,2021",2022,validation
2,validation_2023,"2015,2016,2017,2018,2019,2021,2022",2023,validation
3,validation_2024,"2015,2016,2017,2018,2019,2021,2022,2023",2024,validation
4,final_test_2025,"2015,2016,2017,2018,2019,2021,2022,2023,2024",2025,test


Final test sample (2025):


,season,team_id,regular_season_win_rate,preseason_rating,won_at_least_one_main_bracket_game,data_role
13940,2025,1101,0.482759,8.5,0,test
13941,2025,1102,0.551724,-5.1,0,test
13942,2025,1103,0.464286,2.7,0,test
13943,2025,1104,0.548387,2.7,0,test
13944,2025,1105,0.800000,15.9,1,test
13945,2025,1106,0.275862,0.5,0,test
13946,2025,1107,0.400000,-2.4,0,test
13947,2025,1108,0.571429,-6.2,0,test


Baseline predictors: ['regular_season_win_rate']
Rating-added predictors: ['regular_season_win_rate', 'preseason_rating']
2026 forecast training seasons: [2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023, 2024, 2025]
Saved H2 tables to: C:\Users\lsadi\Documents\GitHub\CSC-4913-Project-4\prepared_data\H2
